# 02 — Preprocessing: Tiling & Mask Generation
**SVAMITVA Hackathon — AI-Based Feature Extraction from Drone Images**

This notebook:
1. Loads data manifest from notebook 01
2. Tiles each orthophoto into 512x512 patches
3. Generates segmentation masks (7 classes) and roof-type masks
4. Saves tiles + metadata for training
5. Visualizes sample tiles for QC

In [ ]:
# ── Cell 1: Setup + Auto-Restore ─────────────────────────────────
import os, sys, json, time, shutil, subprocess
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/IITT_AIML')
    LOCAL_ROOT = Path('/content/IITT_AIML')
else:
    DRIVE_ROOT = Path.home() / 'IITT_AIML'
    LOCAL_ROOT = DRIVE_ROOT

sys.path.insert(0, str(LOCAL_ROOT))

# Make sure src/ is available
if IS_COLAB:
    src_drive = DRIVE_ROOT / 'src'
    src_local = LOCAL_ROOT / 'src'
    if src_drive.exists() and not src_local.exists():
        shutil.copytree(src_drive, src_local)
        print('Copied src/ from Drive')

# ── Auto-restore: re-extract zips if imagery is missing ──
if IS_COLAB:
    imagery_dir = LOCAL_ROOT / 'imagery'
    zips_dir = DRIVE_ROOT / 'zips'
    
    # Check if we need to re-extract
    cg_train = imagery_dir / 'CG_train'
    pb_train = imagery_dir / 'PB_train'
    shp_local = LOCAL_ROOT / 'shp-file'
    
    need_extract = not cg_train.exists() or not pb_train.exists()
    
    if need_extract and zips_dir.exists():
        print('Local imagery missing — re-extracting from Drive zips...')
        
        zip_rules = {
            'CG_Training_dataSet_2.zip': str(cg_train),
            'CG_Training_dataSet_3.zip': str(cg_train),
            'CG_shp-file.zip': str(shp_local),
            'PB_training_dataSet_shp_file.zip': str(pb_train),
        }
        
        for zip_name, dest in zip_rules.items():
            zip_path = zips_dir / zip_name
            if not zip_path.exists():
                continue
            os.makedirs(dest, exist_ok=True)
            print(f'  Extracting {zip_name} -> {dest}')
            result = subprocess.run(
                ['unzip', '-o', '-q', str(zip_path), '-d', dest],
                capture_output=True, text=True
            )
            if result.returncode == 0:
                print(f'  ✓ {zip_name} extracted')
            else:
                print(f'  ✗ {zip_name} FAILED: {result.stderr[:200]}')
        
        # Move PB shapefiles to expected location if needed
        pb_shp_local = pb_train / 'shp-file'
        if pb_train.exists() and (not pb_shp_local.exists() or not list(pb_shp_local.glob('*.shp'))):
            # Find .shp files anywhere in PB_train
            all_shps = list(pb_train.rglob('*.shp'))
            if all_shps:
                shp_parents = set(f.parent for f in all_shps)
                if len(shp_parents) == 1:
                    actual_dir = list(shp_parents)[0]
                    if actual_dir != pb_shp_local:
                        pb_shp_local.mkdir(parents=True, exist_ok=True)
                        for f in actual_dir.iterdir():
                            dest_f = pb_shp_local / f.name
                            if not dest_f.exists():
                                shutil.copy2(f, dest_f)
                        print(f'  Moved PB shapefiles to {pb_shp_local}')
        
        print('Re-extraction complete.')
    elif not need_extract:
        print('Local imagery found — no extraction needed.')

print(f'Project root: {LOCAL_ROOT}')

In [ ]:
# ── Cell 2: Load/Rebuild Data Manifest ────────────────────────────
import rasterio

manifest_path = LOCAL_ROOT / 'data_manifest.json'
if not manifest_path.exists():
    manifest_path = DRIVE_ROOT / 'data_manifest.json'

if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
else:
    manifest = {'working_orthos': [], 'failed_orthos': [], 'shapefile_dirs': {}}

# Verify all ortho paths exist — if not, rebuild manifest from local files
missing = [o for o in manifest['working_orthos'] if not Path(o['path']).exists()]
if missing:
    print(f'{len(missing)} ortho paths stale — rebuilding manifest from local files...')
    
    def find_orthos(d):
        d = Path(d)
        if not d.exists(): return []
        return sorted(list(d.rglob('*.tif')) + list(d.rglob('*.TIF')))
    
    def find_shapefiles(d):
        d = Path(d)
        if not d.exists(): return []
        return sorted(d.rglob('*.shp'))
    
    def validate_ortho(path):
        try:
            with rasterio.open(path) as src:
                win = rasterio.windows.Window(0, 0, min(256, src.width), min(256, src.height))
                src.read(window=win)
                return {
                    'name': path.name, 'path': str(path), 'status': 'OK',
                    'crs': str(src.crs), 'res_cm': round(abs(src.res[0]) * 100, 2),
                    'width': src.width, 'height': src.height, 'bands': src.count,
                }
        except Exception as e:
            return {'name': path.name, 'path': str(path), 'status': f'ERROR: {e}'}
    
    working, failed = [], []
    for state in ['CG_train', 'PB_train']:
        orthos = find_orthos(LOCAL_ROOT / 'imagery' / state)
        for f in orthos:
            info = validate_ortho(f)
            if info['status'] == 'OK':
                working.append({'path': str(f), 'state': state, **info})
            else:
                failed.append({'path': str(f), 'state': state, 'error': info['status']})
    
    # Find shapefile dirs
    shp_dirs = {}
    cg_shp = LOCAL_ROOT / 'shp-file'
    if cg_shp.exists() and find_shapefiles(cg_shp):
        shp_dirs['CG'] = str(cg_shp)
    pb_shp = LOCAL_ROOT / 'imagery' / 'PB_train' / 'shp-file'
    if pb_shp.exists() and find_shapefiles(pb_shp):
        shp_dirs['PB'] = str(pb_shp)
    # Fallback: search inside PB_train
    if 'PB' not in shp_dirs:
        pb_train = LOCAL_ROOT / 'imagery' / 'PB_train'
        pb_shps = find_shapefiles(pb_train)
        if pb_shps:
            shp_dirs['PB'] = str(pb_shps[0].parent)
    
    manifest = {
        'working_orthos': working,
        'failed_orthos': failed,
        'shapefile_dirs': shp_dirs,
    }
    
    # Save updated manifest
    out_path = LOCAL_ROOT / 'data_manifest.json'
    with open(out_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    if IS_COLAB:
        shutil.copy2(out_path, DRIVE_ROOT / 'data_manifest.json')
    print(f'Manifest rebuilt and saved.')

print(f'Working orthophotos: {len(manifest["working_orthos"])}')
for o in manifest['working_orthos']:
    print(f"  {o['name']:50s}  {o['state']}  {o.get('crs','')}  {o.get('res_cm','')}cm")

print(f'\nShapefile directories:')
for state, sdir in manifest['shapefile_dirs'].items():
    print(f'  {state}: {sdir}')

In [ ]:
# ── Cell 3: Configure Tiling ─────────────────────────────────────
from src.config import TileConfig

tile_config = TileConfig(
    tile_size=512,
    overlap=64,               # 64px overlap for context
    min_labeled_ratio=0.01,   # keep tiles with >1% labeled pixels
    bands=(1, 2, 3),          # RGB only (drop alpha)
)

TILES_DIR = str(LOCAL_ROOT / 'tiles')
os.makedirs(TILES_DIR, exist_ok=True)
os.makedirs(os.path.join(TILES_DIR, 'images'), exist_ok=True)
os.makedirs(os.path.join(TILES_DIR, 'masks'), exist_ok=True)

print(f'Tile config: {tile_config.tile_size}x{tile_config.tile_size}, '
      f'overlap={tile_config.overlap}, min_labeled={tile_config.min_labeled_ratio}')
print(f'Output: {TILES_DIR}')

In [ ]:
# ── Cell 4: Tile All Orthophotos ─────────────────────────────────
from src.data_pipeline import load_shapefiles, tile_orthophoto
import rasterio

all_tiles = []
t_start = time.time()

for ortho_info in manifest['working_orthos']:
    ortho_path = ortho_info['path']
    state = ortho_info['state'].replace('_train', '').replace('_test', '')
    
    # Determine which shapefile dir to use
    shp_dir = manifest['shapefile_dirs'].get(state)
    if shp_dir is None:
        print(f'WARNING: No shapefile dir for state {state}, skipping {ortho_info["name"]}')
        continue
    
    # Get orthophoto CRS
    with rasterio.open(ortho_path) as src:
        target_crs = str(src.crs)
    
    # Load & reproject shapefiles to match this orthophoto
    print(f'\n{"="*60}')
    print(f'Processing: {ortho_info["name"]} ({state})')
    print(f'CRS: {target_crs}')
    shapefiles = load_shapefiles(shp_dir, target_crs)
    
    # Extract village name from filename
    # CG examples: BADETUMNAR_450157_BANGAPAL_450155_..._ORTHO.tif
    # PB examples: 28996_NADALA_ORTHO.tif, PINDORI MAYA SINGH-TUGALWAL_28456_ortho.tif
    fname = Path(ortho_path).stem
    # Replace spaces and hyphens with underscores for uniform parsing
    parts = fname.replace(' ', '_').replace('-', '_').split('_')
    # Filter out numeric parts and common suffixes
    name_parts = [p for p in parts if not p.isdigit() 
                  and p.upper() not in ('ORTHO', 'ORI', '3857')]
    village_name = '_'.join(name_parts).lower() if name_parts else fname.lower()
    
    print(f'Village name: {village_name}')
    
    # Tile this orthophoto
    tiles = tile_orthophoto(
        ortho_path, shapefiles, tile_config,
        village_name=village_name,
        output_base=TILES_DIR,
    )
    all_tiles.extend(tiles)

elapsed = time.time() - t_start
print(f'\n{"="*60}')
print(f'TILING COMPLETE')
print(f'{"="*60}')
print(f'Total tiles: {len(all_tiles)}')
print(f'Time: {elapsed/60:.1f} min')

In [ ]:
# ── Cell 5: Save Dataset Metadata ────────────────────────────────
from src.data_pipeline import _compute_class_summary

metadata = {
    'total_tiles': len(all_tiles),
    'tile_size': tile_config.tile_size,
    'overlap': tile_config.overlap,
    'tiles': all_tiles,
    'class_summary': _compute_class_summary(all_tiles),
}

meta_path = os.path.join(TILES_DIR, 'dataset_meta.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

# Copy to Drive for persistence
if IS_COLAB:
    drive_tiles = DRIVE_ROOT / 'tiles'
    drive_tiles.mkdir(parents=True, exist_ok=True)
    shutil.copy2(meta_path, drive_tiles / 'dataset_meta.json')

print(f'Metadata saved: {meta_path}')
print(f'\nClass distribution (tiles containing each class):')
for cls_name, count in metadata['class_summary'].items():
    pct = count / len(all_tiles) * 100 if all_tiles else 0
    print(f'  {cls_name:12s}: {count:5d} tiles ({pct:5.1f}%)')

In [ ]:
# ── Cell 6: Visualize Sample Tiles (QC) ──────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from src.config import SEG_CLASSES, SEG_COLORS

def visualize_tiles(tiles_dir, n_samples=8, seed=42):
    """Show random tile-mask pairs for quality check."""
    images_dir = os.path.join(tiles_dir, 'images')
    masks_dir = os.path.join(tiles_dir, 'masks')
    
    image_files = sorted(Path(images_dir).glob('*.npy'))
    if not image_files:
        print('No tiles found!'); return
    
    np.random.seed(seed)
    indices = np.random.choice(len(image_files), min(n_samples, len(image_files)), replace=False)
    
    fig, axes = plt.subplots(2, n_samples, figsize=(3*n_samples, 6))
    if n_samples == 1:
        axes = axes.reshape(2, 1)
    
    # Build color map
    color_map = np.zeros((max(SEG_COLORS.keys())+1, 3), dtype=np.uint8)
    for cls_id, color in SEG_COLORS.items():
        color_map[cls_id] = color
    
    for j, idx in enumerate(indices):
        img_path = image_files[idx]
        mask_path = Path(masks_dir) / img_path.name.replace('.npy', '_seg.npy')
        
        img = np.load(img_path)
        if img.shape[0] <= 4:  # CHW
            img = img.transpose(1, 2, 0)
        img_rgb = img[:, :, :3]
        
        axes[0, j].imshow(img_rgb)
        axes[0, j].set_title(img_path.stem[:20], fontsize=8)
        axes[0, j].axis('off')
        
        if mask_path.exists():
            mask = np.load(mask_path)
            mask_rgb = color_map[mask]
            axes[1, j].imshow(mask_rgb)
            # Show class counts
            classes_present = np.unique(mask)
            class_names = [SEG_CLASSES.get(c, f'{c}') for c in classes_present if c > 0]
            axes[1, j].set_title(', '.join(class_names), fontsize=7)
        axes[1, j].axis('off')
    
    axes[0, 0].set_ylabel('Image', fontsize=12)
    axes[1, 0].set_ylabel('Mask', fontsize=12)
    plt.tight_layout()
    plt.savefig(str(LOCAL_ROOT / 'tile_samples.png'), dpi=150, bbox_inches='tight')
    plt.show()

visualize_tiles(TILES_DIR)
print('\nProceed to 03_train.ipynb')

In [ ]:
# ── Cell 7: Copy tiles to Drive for persistence (Colab) ─────────
if IS_COLAB:
    drive_tiles = DRIVE_ROOT / 'tiles'
    drive_images = drive_tiles / 'images'
    drive_masks = drive_tiles / 'masks'
    drive_images.mkdir(parents=True, exist_ok=True)
    drive_masks.mkdir(parents=True, exist_ok=True)
    
    import glob
    local_images = glob.glob(os.path.join(TILES_DIR, 'images', '*.npy'))
    local_masks = glob.glob(os.path.join(TILES_DIR, 'masks', '*.npy'))
    
    print(f'Copying {len(local_images)} image tiles + {len(local_masks)} mask tiles to Drive...')
    for f in local_images:
        shutil.copy2(f, drive_images / Path(f).name)
    for f in local_masks:
        shutil.copy2(f, drive_masks / Path(f).name)
    print('Done. Tiles saved to Drive for persistence across sessions.')
else:
    print('Local mode: tiles already on disk.')